# Task 2 — Chunking & Vector Store
### TechNova Support · RAG Project


## 1. Imports & Config

In [16]:
import pathlib, random
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

load_dotenv(override=True)

EMBED_MODEL   = "all-MiniLM-L6-v2"
DB_NAME       = "vector_db"    
CHUNK_SIZE    = 700
CHUNK_OVERLAP = 120
TOP_K         = 5

print(f"Embed model : {EMBED_MODEL}")
print(f"Vector DB   : {DB_NAME}")

Embed model : all-MiniLM-L6-v2
Vector DB   : vector_db


## 2. Load Documents from Task 1

In [8]:
# Re-run Task 1 so this notebook is self-contained
%run Task1_Ingestion.ipynb
print(f"\nLoaded {len(raw_docs)} documents")

Imports OK
Loaded 34 documents from knowledge-base/

  [company             ]  company\about.md
  [company             ]  company\careers.md
  [company             ]  company\contact.md
  [company             ]  company\sustainability.md
  [faqs                ]  faqs\account.md
  [faqs                ]  faqs\payments.md
  [faqs                ]  faqs\returns.md
  [faqs                ]  faqs\shipping.md
  [faqs                ]  faqs\software_updates.md
  [faqs                ]  faqs\trade_in.md
  [faqs                ]  faqs\troubleshooting.md
  [faqs                ]  faqs\warranty.md
  [policies            ]  policies\privacy_policy.md
  [policies            ]  policies\return_policy.md
  [policies            ]  policies\shipping_policy.md
  [policies            ]  policies\terms_of_service.md
  [policies            ]  policies\warranty_policy.md
  [products            ]  products\nova_earbuds.md
  [products            ]  products\nova_laptop_pro.md
  [products            ]  produc

## 3. Chunking


In [17]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size     = CHUNK_SIZE,
    chunk_overlap  = CHUNK_OVERLAP,
    separators     = ["\n\n", "\n", ". ", " ", ""],
    add_start_index= True,
)

chunks = splitter.split_documents(raw_docs)
print(f"Produced {len(chunks)} chunks from {len(raw_docs)} documents.")

for c in random.sample(chunks, 2):
    print(f"\n  source   : {c.metadata['source']}")
    print(f"  category : {c.metadata['category']}")
    print(f"  length   : {len(c.page_content)} chars")
    print(f"  preview  : {c.page_content[:120].strip()!r}")

Produced 104 chunks from 34 documents.

  source   : company\careers.md
  category : company
  length   : 39 chars
  preview  : '## Contact\ncareers@technova.example.com'

  source   : faqs\account.md
  category : faqs
  length   : 478 chars
  preview  : 'Order history is retained for 7 years for legal/tax reasons but is anonymized.\n\n## My account was locked. Why?\nCommon re'


## 4. Embeddings & Persisted Vector Store


In [18]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)


def build_or_load_vectorstore(chunks, db_name=DB_NAME, force_rebuild=False):
    """Load existing Chroma if chunk count matches, else rebuild."""
    if not force_rebuild and pathlib.Path(db_name).exists():
        vs    = Chroma(persist_directory=db_name, embedding_function=embeddings)
        count = vs._collection.count()
        if count == len(chunks):
            print(f"✓ Loaded existing vector store ({count} chunks). Skipping rebuild.")
            return vs
        print(f"Chunk count mismatch (DB={count} vs expected={len(chunks)}). Rebuilding...")

    print(f"Building vector store with {len(chunks)} chunks — this takes ~30–60 seconds...")
    vs = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=db_name
    )
    print(f"✓ Done. Collection size: {vs._collection.count()}")
    return vs


vectorstore = build_or_load_vectorstore(chunks)
print(f"\nFinal collection size : {vectorstore._collection.count()}")
sample = vectorstore._collection.get(limit=1, include=["metadatas"])
print(f"Sample metadata       : {sample['metadatas'][0]}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✓ Loaded existing vector store (104 chunks). Skipping rebuild.

Final collection size : 104
Sample metadata       : {'category': 'company', 'start_index': 0, 'source': 'company\\about.md'}


## 5. Retrieval with Metadata Filtering

In [19]:
def retrieve(question: str, k: int = TOP_K, category: str = None) -> list:
    """
    Return top-k chunks semantically closest to `question`.
    Optional `category` filter restricts search to one subfolder.
    """
    search_kwargs = {"k": k}
    if category and category not in (None, "all", "All"):
        search_kwargs["filter"] = {"category": category}
    return vectorstore.as_retriever(search_kwargs=search_kwargs).invoke(question)


tests = [
    ("What is the return window for NovaCare+ members?", "policies"),
    ("How much is the Nova Phone X1?",                  None),
    ("cracked watch screen ticket",                     None),
    ("purple banana spaceship",                         None),
]

print('Retrieval sanity checks:')
for q, cat in tests:
    docs    = retrieve(q, k=3, category=cat)
    sources = [d.metadata['source'] for d in docs]
    print(f"\n  Q: {q[:55]!r}")
    print(f"  filter={cat}")
    print(f"  sources={sources}")

Retrieval sanity checks:

  Q: 'What is the return window for NovaCare+ members?'
  filter=policies
  sources=['policies\\return_policy.md', 'policies\\warranty_policy.md', 'policies\\return_policy.md']

  Q: 'How much is the Nova Phone X1?'
  filter=None
  sources=['products\\nova_phone_x1.md', 'products\\nova_phone_x1.md', 'products\\nova_tablet.md']

  Q: 'cracked watch screen ticket'
  filter=None
  sources=['support_tickets\\ticket_004.md', 'support_tickets\\ticket_004.md', 'support_tickets\\ticket_008.md']

  Q: 'purple banana spaceship'
  filter=None
  sources=['products\\nova_speaker.md', 'products\\nova_phone_x1.md', 'products\\nova_speaker.md']
